In [ ]:
!wget -q http://www.statmt.org/europarl/v7/fr-en.tgz
!tar -xzf fr-en.tgz
!pip install -q sacrebleu
print('Done.')

In [ ]:
from collections import defaultdict
import sacrebleu

def tokenize(s):
    return s.lower().strip().split()

with open('europarl-v7.fr-en.fr', encoding='utf-8') as f:
    fr_raw = [l.strip() for l in f]
with open('europarl-v7.fr-en.en', encoding='utf-8') as f:
    en_raw = [l.strip() for l in f]

# Drop empty lines
pairs = [(f, e) for f, e in zip(fr_raw, en_raw) if f and e]
fr_raw, en_raw = [p[0] for p in pairs], [p[1] for p in pairs]
n = len(fr_raw)
print(f'Total sentence pairs: {n:,}')

In [ ]:
# Koehn-style test split: last 1,755 sentences with French token length 5-15.
# This matches the protocol used by the phrase-based system in Phrase_model_new.ipynb.
TRAIN_SIZE        = 50000
EUROPARL_TEST_SIZE = 1755
TEST_SRC_MIN_LEN  = 5
TEST_SRC_MAX_LEN  = 15

qual_idx = [
    i for i in range(n)
    if TEST_SRC_MIN_LEN <= len(tokenize(fr_raw[i])) <= TEST_SRC_MAX_LEN
]
test_idx_set = set(qual_idx[-EUROPARL_TEST_SIZE:])
train_idx    = [i for i in range(n) if i not in test_idx_set][:TRAIN_SIZE]

test_order   = sorted(test_idx_set)
test_fr_tok  = [tokenize(fr_raw[i]) for i in test_order]
test_en_ref  = [en_raw[i] for i in test_order]

train_fr = [tokenize(fr_raw[i]) for i in train_idx]
train_en = [tokenize(en_raw[i]) for i in train_idx]

print(f'Training pairs : {len(train_fr):,}')
print(f'Test pairs     : {len(test_fr_tok):,}')
print(f'Sample test FR : {" ".join(test_fr_tok[0])}')
print(f'Sample test EN : {test_en_ref[0]}')

In [ ]:
# Initialize translation table uniformly over co-occurring word pairs.
fr_to_en = defaultdict(set)
for fr_s, en_s in zip(train_fr, train_en):
    for f in fr_s:
        for e in en_s:
            fr_to_en[f].add(e)

t = defaultdict(float)
for f, en_words in fr_to_en.items():
    for e in en_words:
        t[(e, f)] = 1.0 / len(en_words)

print(f'Translation table entries: {len(t):,}')

In [ ]:
# 10 EM iterations (same as ibm_model1.ipynb).
for i in range(10):
    count = defaultdict(float)
    total = defaultdict(float)
    for en_s, fr_s in zip(train_en, train_fr):
        for e in en_s:
            s = sum(t[(e, f)] for f in fr_s)
            if s == 0:
                continue
            for f in fr_s:
                d = t[(e, f)] / s
                count[(e, f)] += d
                total[f]      += d
    for (e, f) in count:
        if total[f]:
            t[(e, f)] = count[(e, f)] / total[f]
    print(f'Iteration {i+1}/10')
print('Training done.')

In [ ]:
# Build greedy best-translation lookup and evaluate.
best = {}
for (e, f), p in t.items():
    if f not in best or p > best[f][1]:
        best[f] = (e, p)

def translate(tokens):
    return ' '.join(best[w][0] if w in best else '<UNK>' for w in tokens)

translations = [translate(s) for s in test_fr_tok]

bleu = sacrebleu.corpus_bleu(translations, [test_en_ref], lowercase=True)
print(f'IBM Model 1 Europarl in-domain BLEU ({len(test_fr_tok)} sentences): {bleu.score:.2f}')

print('\nSample translations:')
for i in range(5):
    print(f'FR:  {" ".join(test_fr_tok[i])}')
    print(f'REF: {test_en_ref[i]}')
    print(f'IBM: {translations[i]}')
    print()